### Text Parsing:

Using PdfPlumber to extract the text from the PDF

In [9]:
import pdfplumber

def extract_text_from_pdf(file_path):
    text = ""

    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"

    return text

In [14]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [116]:
def read_txt(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return f.read()

In [15]:
def parse_resume(file_path):
    raw_text = extract_text_from_pdf(file_path)
    clean = clean_text(raw_text)
    return clean

### Text Preprocessing

Using Spacy to preprocess the text

In [18]:
import spacy

nlp = spacy.load("en_core_web_sm")

In [122]:
def extract_keywords(text: str):
    doc = nlp(text)
    
    keywords = set()
    
    for token in doc:
        if token.pos_ in ["NOUN", "PROPN", "ADJ"]:
            keywords.add(token.lemma_.lower())

    return keywords

In [130]:
def extract_phrases(text, keywords):
    doc = nlp(text)
    
    phrases = []

    phrases = [
        chunk.text.lower()
        for chunk in doc.noun_chunks
    ]

    phrases = list(filter(lambda phrase: phrase not in keywords, phrases))

    return phrases

### Embeddings (Vectorization)

Using Sentence-Transformers to get the embeddings

In [44]:
from sentence_transformers import SentenceTransformer

c:\Users\sayan\OneDrive\Desktop\Learning\virtusa\projects\resume-analyser\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [45]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 121.38it/s]


In [46]:
def get_embeddings(text_list):
    return model.encode(text_list)

In [ ]:
# Getting the score:

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def get_score(resume_embeddings, jd_embeddings):
    score_matrix = cosine_similarity(
        resume_embeddings, jd_embeddings
    )

    best_matches = np.max(score_matrix, axis=1)

    return best_matches.mean()

In [158]:
text = parse_resume("resume2.pdf")
resume_keywords = extract_keywords(text)

resume_phrases = extract_phrases(text, resume_keywords)

resume_embeddings = get_embeddings(resume_phrases)

In [160]:
jd = read_txt("sample_jd.txt")
cleaned_jd = clean_text(jd)

jd_keywords = extract_keywords(cleaned_jd)

jd_phrases = extract_phrases(cleaned_jd, jd_keywords)
jd_embeddings = get_embeddings(jd_phrases)

In [167]:
resume_keyword_embeddings = get_embeddings(list(resume_keywords))
jd_keyword_embeddings = get_embeddings(list(jd_keywords))

In [168]:
keywords_embed_score = get_score(resume_keyword_embeddings, jd_keyword_embeddings)

In [161]:
common_skills = resume_keywords & jd_keywords

missing_skills = jd_keywords - resume_keywords
extra_skills = resume_keywords - jd_keywords

In [162]:
semantic_score = get_score(resume_embeddings, jd_embeddings)
keyword_score = len(common_skills) / len(jd_keywords)

In [163]:
final_score = (0.7 * semantic_score) + (0.3 * keyword_score)

In [165]:
missing_skills

{'):',
 'analytical',
 'assist',
 'basic',
 'bonus',
 'candidate',
 'case',
 'clean',
 'concept',
 'contribution',
 'criterion',
 'debug',
 'degree',
 'deploying',
 'developer',
 'eager',
 'eligibility',
 'engineer',
 'familiarity',
 'field',
 'foundation',
 'fresher',
 'frontend',
 'ideal',
 'integration',
 'key',
 'level',
 'maintainable',
 'participation',
 'performance',
 'platform',
 'powered',
 'practice',
 'proficiency',
 'profile',
 'responsibility',
 'role',
 'seamless',
 'solution',
 'such',
 'test',
 'use',
 'work'}

In [170]:
print("With keyword embeddings")
print(f"Semantic score: {semantic_score * 100 :.2f}")
print(f"Keyword score: {keywords_embed_score * 100 :.2f}")
print(f"Final score: {((0.7 * semantic_score) + (0.3 * keywords_embed_score)) * 100 :.2f}")

With keyword embeddings
Semantic score: 59.42
Keyword score: 71.62
Final score: 63.08


In [ ]:
print("With modified JD")
print(f"Semantic score: {semantic_score * 100 :.2f}")
print(f"Keyword score: {keyword_score * 100 :.2f}")
print(f"Final score: {final_score * 100 :.2f}")

With modified JD
Semantic score: 59.42
Keyword score: 62.93
Final score: 60.47


In [156]:
print(f"Semantic score: {semantic_score * 100 :.2f}")
print(f"Keyword score: {keyword_score * 100 :.2f}")
print(f"Final score: {final_score * 100 :.2f}")

Semantic score: 60.04
Keyword score: 57.86
Final score: 59.38


In [151]:
print(f"Semantic score: {semantic_score * 100 :.2f}")
print(f"Keyword score: {keyword_score * 100 :.2f}")
print(f"Final score: {final_score * 100 :.2f}")

Semantic score: 58.23
Keyword score: 37.14
Final score: 51.90


In [146]:
print(f"Semantic score: {semantic_score * 100 :.2f}")
print(f"Keyword score: {keyword_score * 100 :.2f}")
print(f"Final score: {final_score * 100 :.2f}")

Semantic score: 52.20
Keyword score: 45.71
Final score: 50.25
